# Import libraries

In [2]:
import os
import warnings
warnings.filterwarnings("ignore") # remove some scikit-image warnings

from monai.apps import DecathlonDataset
from monai.data import DataLoader
from monai.transforms import (
    LoadImageD,
    Compose,
    LoadImageD,
)

import torch
import numpy as np
import matplotlib.pyplot as plt
import random

#### Set seeds

In [3]:
import sys
from tqdm import tqdm
import pickle as pkl
import glob
import json
import nibabel as nib

In [4]:
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

In [5]:
# msd_all_tasks_path = f'/raid/CataChiru/MedicalDecathlonAllTasks/'
msd_all_tasks_path = f'./datasets/MSD/MedicalDecathlon'

unused_tasks = glob.glob(f'{msd_all_tasks_path}/*/')

In [6]:
blacklisted_files ={
    'Task03_Liver': [ 'liver_141.nii.gz', 'liver_156.nii.gz', 'liver_160.nii.gz', 'liver_161.nii.gz', 'liver_162.nii.gz', 'liver_164.nii.gz', 'liver_167.nii.gz', 'liver_182.nii.gz', 'liver_189.nii.gz', 'liver_190.nii.gz'],
    'Task08_HepaticVessel': ['hepaticvessel_247.nii.gz']
}

def keep_only_names(names_lists):
    for i in range(len(names_lists)):
        names_lists[i] = names_lists[i].split('/')[-1]
    return names_lists


def create_empty_prediction_for_task(task_path : str, model_name : str):
    print(f'Creating empty predictions for {task_path} and {model_name}')

    task_name = task_path.split('/')[-2]

    with open(f'{task_path}/dataset.json') as crt_file:
        dataset_json = json.load(crt_file)

    # Remove '/imagesTs/' from the names
    # test_names list contains: ['lung_002.nii.gz', 'lung_007.nii.gz', ...]
    test_names = keep_only_names(dataset_json['test'])

    if task_name in blacklisted_files:
        print(f'Task {task_name} has blacklisted files, original length: {len(test_names)}')
        test_names = list(set(test_names) - set(blacklisted_files[task_name]))
        print(f'After removing blacklisted files, new length: {len(test_names)}')

    for i in range(len(test_names)):
        original_image = nib.load(f'{task_path}/imagesTs/{test_names[i]}')
        crt_prediction = np.zeros(original_image.shape)
        final_prediction = nib.Nifti1Image(crt_prediction, original_image.affine, original_image.header)

        os.makedirs(f'./predictions/{model_name.upper()}/{task_name}', exist_ok=True)

        nib.save(final_prediction, f'./predictions/{model_name.upper()}/{task_name}/{test_names[i]}')



def create_empty_predictions_for_all_tasks(model_name : str):
    for task_path in unused_tasks:
        assert "Task06_Lung" not in task_path, "Task06_Lung should have good predictions in it"
        create_empty_prediction_for_task(task_path, model_name)



create_empty_predictions_for_all_tasks('empty_submission')

Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Task10_Colon/ and empty_submission
Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Task04_Hippocampus/ and empty_submission
Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Task03_Liver/ and empty_submission
Task Task03_Liver has blacklisted files, original length: 70
After removing blacklisted files, new length: 60
Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Task02_Heart/ and empty_submission
Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Task08_HepaticVessel/ and empty_submission
Task Task08_HepaticVessel has blacklisted files, original length: 139
After removing blacklisted files, new length: 139
Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Task09_Spleen/ and empty_submission
Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Task05_Prostate/ and empty_submission
Creating empty predictions for ./datasets/MSD/MedicalDecathlon/Tas

In [27]:
task_path = '/home/aimas/Desktop/Hackathon_Enfield/datasets/MSD/MedicalDecathlon/Task06_Lung'

task_name = task_path.split('/')[-2]

with open(f'{task_path}/dataset.json') as crt_file:
       dataset_json = json.load(crt_file)

# Remove '/imagesTs/' from the names
# test_names list contains: ['lung_002.nii.gz', 'lung_007.nii.gz', ...]
test_names = keep_only_names(dataset_json['test'])

for i in range(len(test_names)):
       original_image = nib.load(f'{task_path}/imagesTs/{test_names[i]}')
       print(original_image.affine)
       print(original_image.header)
       break
       
       #TODO: Printat original_image.affine pe lung dupa, sa vad ce e aici



[[  -0.78125      0.           0.         200.       ]
 [   0.           0.78125      0.        -217.8187561]
 [   0.           0.           0.625     -313.5      ]
 [   0.           0.           0.           1.       ]]
<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : b''
db_name         : b''
extents         : 0
session_error   : 0
regular         : b'r'
dim_info        : 0
dim             : [  3 512 512 271   1   1   1   1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : float32
bitpix          : 32
slice_start     : 0
pixdim          : [-1.       0.78125  0.78125  0.625    0.       0.       0.       0.     ]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 10
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toffset         : 0.0
glmax           : 0
glmin           : 0
descrip     